<a href="https://colab.research.google.com/github/gurnisha-rehal/group3-blink-detector/blob/main/blink_detector_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 3 Blink Detection - Computer Vision Project

## Cloning YOLO

### Connect Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd "/content/drive/MyDrive/Colab Notebooks"
!ls

/content/drive/MyDrive/Colab Notebooks
blink_detector_training.ipynb  ultralytics	Untitled1.ipynb  yolov8s.pt
runs			       Untitled0.ipynb	yolov8n.pt	 zidane.jpg


### Install YOLOv8

In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 14.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [4]:
import ultralytics
ultralytics.checks()

from ultralytics import YOLO

Ultralytics 8.4.199 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 0.1/112.6 GB disk)


In [5]:
!git clone https://github.com/ultralytics/ultralytics.git

fatal: destination path 'ultralytics' already exists and is not an empty directory.


### Extract images

In [12]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/IAT360 CV Project/blink-detector/dataset.zip"

source_dir = "/content/drive/MyDrive/IAT360 CV Project/blink-detector/coco_source"

os.makedirs(source_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(source_dir)

print("Extraction complete")
print(os.listdir(source_dir)[:10])

Extraction complete
['test', 'train', 'valid', 'README.roboflow.txt']


### Verify categories and classes

In [14]:
for folder in ["train", "valid", "test"]:
    path = os.path.join(source_dir, folder)
    files = os.listdir(path)

    images = [f for f in files if f.endswith(('.jpg', '.jpeg', '.png'))]
    annotations = [f for f in files if f.endswith('.json')]

    print(f"\n{folder}:")
    print("Images:", len(images))
    print("Annotations:", annotations)


train:
Images: 77
Annotations: ['_annotations.coco.json']

valid:
Images: 9
Annotations: ['_annotations.coco.json']

test:
Images: 10
Annotations: ['_annotations.coco.json']


In [15]:
import json

annotation_path = os.path.join(source_dir, "train", "_annotations.coco.json")

with open(annotation_path, "r") as f:
    data = json.load(f)

print("Categories:")
for category in data["categories"]:
    print(category)

Categories:
{'id': 0, 'name': 'eyed', 'supercategory': 'none'}
{'id': 1, 'name': 'closed', 'supercategory': 'eyed'}
{'id': 2, 'name': 'open', 'supercategory': 'eyed'}


#### Create data.yaml file

In [26]:
%%writefile "/content/drive/MyDrive/IAT360 CV Project/blink-detector/dataset/data.yaml"
path: /content/drive/MyDrive/IAT360 CV Project/blink-detector/dataset

train: images/train
val: images/val
test: images/test

names:
  0: Open
  1: Closed

Writing /content/drive/MyDrive/IAT360 CV Project/blink-detector/dataset/data.yaml


In [16]:
print("Number of images:", len(data["images"]))
print("Number of annotations:", len(data["annotations"]))

print("\nSample annotation:")
print(data["annotations"][0])

Number of images: 77
Number of annotations: 101

Sample annotation:
{'id': 1, 'image_id': 0, 'category_id': 2, 'bbox': [404.78, 400, 281.22, 80], 'iscrowd': 0, 'area': 22497.6, 'segmentation': []}


### Covert Roboflow COCO to YOLOv8

In [24]:
import os

source_dir = "/content/drive/MyDrive/IAT360 CV Project/blink-detector/coco_source"
output_dir = "/content/drive/MyDrive/IAT360 CV Project/blink-detector/dataset"

# Check original folders
for split in ["train", "valid", "test"]:
    path = os.path.join(source_dir, split)
    print(split, "exists:", os.path.exists(path))
# Create YOLO folder structure
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(output_dir, "images", split), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", split), exist_ok=True)

print("\nYOLO folders created")

train exists: True
valid exists: True
test exists: True

YOLO folders created


In [25]:
import json
import shutil
from collections import Counter

# Map original COCO category IDs to YOLO class IDs
category_mapping = {
    2: 0,  # Open
    1: 1   # Closed
}

# Map original folders to YOLO folders
split_mapping = {
    "train": "train",
    "valid": "val",
    "test": "test"
}

for original_split, yolo_split in split_mapping.items():

    split_dir = os.path.join(source_dir, original_split)
    annotation_path = os.path.join(split_dir, "_annotations.coco.json")

    # Load COCO annotations
    with open(annotation_path, "r") as f:
        coco_data = json.load(f)

    # Index annotations by image ID
    annotations_by_image = {}

    for annotation in coco_data["annotations"]:
        image_id = annotation["image_id"]
        annotations_by_image.setdefault(image_id, []).append(annotation)

    converted_count = Counter()
    image_count = 0

    for image in coco_data["images"]:

        image_id = image["id"]
        filename = image["file_name"]

        image_path = os.path.join(split_dir, filename)

        # Skip missing images
        if not os.path.exists(image_path):
            print("Missing image:", image_path)
            continue

        # Copy image into YOLO dataset
        destination_image = os.path.join(
            output_dir, "images", yolo_split, os.path.basename(filename)
        )
        shutil.copy2(image_path, destination_image)

        # Create corresponding label file
        label_filename = os.path.splitext(os.path.basename(filename))[0] + ".txt"
        label_path = os.path.join(
            output_dir, "labels", yolo_split, label_filename
        )

        labels = []

        for annotation in annotations_by_image.get(image_id, []):

            category_id = annotation["category_id"]

            # Ignore parent category "eyed"
            if category_id not in category_mapping:
                continue

            x, y, width, height = annotation["bbox"]

            image_width = image["width"]
            image_height = image["height"]

            # Convert COCO bounding box to YOLO format
            x_center = (x + width / 2) / image_width
            y_center = (y + height / 2) / image_height
            normalized_width = width / image_width
            normalized_height = height / image_height

            class_id = category_mapping[category_id]

            labels.append(
                f"{class_id} {x_center:.6f} {y_center:.6f} "
                f"{normalized_width:.6f} {normalized_height:.6f}"
            )

            converted_count[class_id] += 1

        # Write labels (even if no objects are annotated)
        with open(label_path, "w") as f:
            f.write("\n".join(labels))

        image_count += 1

    print(f"\n{yolo_split}:")
    print("Images copied:", image_count)
    print("Open annotations:", converted_count[0])
    print("Closed annotations:", converted_count[1])

print("\nCOCO to YOLO conversion complete!")


train:
Images copied: 77
Open annotations: 86
Closed annotations: 15

val:
Images copied: 9
Open annotations: 11
Closed annotations: 1

test:
Images copied: 10
Open annotations: 11
Closed annotations: 4

COCO to YOLO conversion complete!


In [27]:
for split in ["train", "val", "test"]:

    image_dir = os.path.join(output_dir, "images", split)
    label_dir = os.path.join(output_dir, "labels", split)

    images = [f for f in os.listdir(image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    labels = [f for f in os.listdir(label_dir) if f.endswith(".txt")]

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))


TRAIN
Images: 77
Labels: 77

VAL
Images: 9
Labels: 9

TEST
Images: 10
Labels: 10


### Testing

In [ ]:
!yolo predict model=yolov8n.pt source='https://ultralytics.com/images/zidane.jpg' save=True

Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

Found https://ultralytics.com/images/zidane.jpg locally at zidane.jpg
image 1/1 /content/drive/MyDrive/Colab Notebooks/zidane.jpg: 384x640 2 persons, 1 tie, 9.0ms
Speed: 89.7ms preprocess, 9.0ms inference, 36.5ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/drive/MyDrive/Colab Notebooks/runs/detect/predict-2
💡 Learn more at https://docs.ultralytics.com/modes/predict


## Prepared Training

In [ ]:
model = YOLO('yolov8s.pt')

# replace with actual location of data.yaml file
# adjust hyperparameters after the baseline run
results = model.train(
    data='/content/drive/MyDrive/blink-detector/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    patience=10,
    project='/content/drive/MyDrive/blink-detector/runs',
    name='blink_detection'
)

## Validate model accuracy

In [ ]:
metrics = model.val()

### Test on images



In [ ]:
model = YOLO("/content/runs/detect/blink_detection/weights/best.pt")

results = model.predict(
    source="/content/drive/MyDrive/blink-detector/test_image.jpg",
    conf=0.25,
    save=True,
    show_labels=True
)